In [3]:
!pip install mp-api
apikey = "4Fxu8kCP9i4QlEKWL4gUCKY8WfTujzM8"
from mp_api.client import MPRester

def calculate_vickers_hardness(material_id):
    with MPRester(apikey) as mpr:
        try:
            data = mpr.materials.summary.search(
                material_ids=[material_id], 
                fields=["formula_pretty", "bulk_modulus", "shear_modulus", "density"]
            )
            
            if not data:
                return f"No material found for ID: {material_id}"
            
            material = data[0]

            b_dict = material.bulk_modulus
            g_dict = material.shear_modulus
            d = material.density
            
            if b_dict is None or g_dict is None:
                return f"No elastic data available for {material.formula_pretty}."

            B = b_dict.get('vrh') if isinstance(b_dict, dict) else b_dict
            G = g_dict.get('vrh') if isinstance(g_dict, dict) else g_dict
            
            if B is None or G is None:
                return "Could not find VRH average in the elastic data."

            # Pugh Ratio
            k = G / B
            
            # Chen-Pugh Vickers Hardness Formula: Hv = 2 * (k^2 * G)^0.585 - 3
            #H5 in the Vickers paper
            hardness = 2 * (pow(k**2 * G, 0.585)) - 3
            
            hv_final = max(0, hardness)
            
            return {
                "Material": material.formula_pretty,
                "Density": f"{d:.2f} m^3/kg",
                "Bulk Modulus (B)": f"{B:.2f} GPa",
                "Shear Modulus (G)": f"{G:.2f} GPa",
                "Pugh Ratio": f"{k:.3f}",
                "Vickers Hardness": f"{hv_final:.2f} GPa",
                "Vickers (kgf/mm2)": f"{hv_final * 101.97:.2f}"
            }

        except Exception as e:
            return f"An error occurred: {e}"

results = calculate_vickers_hardness("mp-22862")
print(results)

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

{'Material': 'NaCl', 'Density': '2.22 m^3/kg', 'Bulk Modulus (B)': '23.77 GPa', 'Shear Modulus (G)': '14.30 GPa', 'Pugh Ratio': '0.602', 'Vickers Hardness': '2.23 GPa', 'Vickers (kgf/mm2)': '227.49'}
